In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

customers_data = [
    (1, "Alice", 30, "USA"),
    (2, "Bob", 25, "Canada"),
    (3, "Cathy", 35, "UK"),
    (4, "David", 28, "India"),
    (5, "Eve", 40, "Germany"),
    (6, "Frank", 32, "Australia"),
    (7, "Grace", 29, "France"),
    (8, "Hank", 33, "USA"),
    (9, "Ivy", 27, "India"),
    (10, "Jack", 36, "Canada")
]
purchases_data = [
    (101, 1, "Laptop", 1200),
    (102, 2, "Smartphone", 800),
    (103, 3, "Tablet", 600),
    (104, 4, "Headphones", 200),
    (105, 5, "Monitor", 300),
    (106, 6, "Keyboard", 100),
    (107, 7, "Mouse", 50),
    (108, 8, "Printer", 150),
    (109, 9, "Camera", 500),
    (110, 1, "Smartwatch", 250)
]

purchases_schema = ["purchase_id", "customer_id", "item", "price"]
customers_schema = ["customer_id", "name", "age", "country"]
customers_df = spark.createDataFrame(data=customers_data, schema=customers_schema)
purchases_df = spark.createDataFrame(data=purchases_data, schema=purchases_schema)


customers_df.show()
purchases_df.show()

+-----------+-----+---+---------+
|customer_id| name|age|  country|
+-----------+-----+---+---------+
|          1|Alice| 30|      USA|
|          2|  Bob| 25|   Canada|
|          3|Cathy| 35|       UK|
|          4|David| 28|    India|
|          5|  Eve| 40|  Germany|
|          6|Frank| 32|Australia|
|          7|Grace| 29|   France|
|          8| Hank| 33|      USA|
|          9|  Ivy| 27|    India|
|         10| Jack| 36|   Canada|
+-----------+-----+---+---------+

+-----------+-----------+----------+-----+
|purchase_id|customer_id|      item|price|
+-----------+-----------+----------+-----+
|        101|          1|    Laptop| 1200|
|        102|          2|Smartphone|  800|
|        103|          3|    Tablet|  600|
|        104|          4|Headphones|  200|
|        105|          5|   Monitor|  300|
|        106|          6|  Keyboard|  100|
|        107|          7|     Mouse|   50|
|        108|          8|   Printer|  150|
|        109|          9|    Camera|  500|
|      

In [0]:
# Performing inner join on costumer and purchases dfs
customers_df.join(purchases_df, customers_df['customer_id']==purchases_df['customer_id'], 'inner').display(20)

customer_id,name,age,country,purchase_id,customer_id,item,price
1,Alice,30,USA,101,1,Laptop,1200
1,Alice,30,USA,110,1,Smartwatch,250
2,Bob,25,Canada,102,2,Smartphone,800
3,Cathy,35,UK,103,3,Tablet,600
4,David,28,India,104,4,Headphones,200
5,Eve,40,Germany,105,5,Monitor,300
6,Frank,32,Australia,106,6,Keyboard,100
7,Grace,29,France,107,7,Mouse,50
8,Hank,33,USA,108,8,Printer,150
9,Ivy,27,India,109,9,Camera,500


In [0]:
# If i want to select customer_id directly from the above table, then it will give "ambiguity error". Since, program confuse which customer id to pick. Unless we specify the name of the table.

customers_df.join(purchases_df, customers_df['customer_id']==purchases_df['customer_id'], 'inner')\
    .select("customer_id").display()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-91693912517835>:3
      1 # If i want to select customer_id directly from the above table, then it will give "ambiguity error". Since, program confuse which customer id to pick. Unless we specify the name of the table.
----> 3 customers_df.join(purchases_df, customers_df['customer_id']==purchases_df['customer_id'], 'inner')\
      4     .select("customer_id").display()

File /databricks/spark/python/pyspark/instrumentation_utils.py:48, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     46 start = time.perf_counter()
     47 try:
---> 48     res = func(*args, **kwargs)
     49     logger.log_success(
     50         module_name, class_name, function_name, time.perf_counter() - start, signature
     51     )
     52     return res

File /databricks/spark/python/pyspark/sql/dataframe.py:3023, in DataFrame.select(self, *

In [0]:
customers_df.join(purchases_df, customers_df['customer_id']==purchases_df['customer_id'], 'inner')\
    .select(purchases_df["customer_id"]).display()

customer_id
1
1
2
3
4
5
6
7
8
9


In [0]:
# Now, I'll filter out those records who didn't purchase anything
notpurchased = customers_df.join(purchases_df,customers_df['customer_id']==purchases_df['customer_id'],'left')
notpurchased.withColumn('Purchased',when(col('purchase_id').isNull(),'No')
                        .otherwise('Yes'))\
                        .select(customers_df['customer_id'],'Purchased').distinct().display()



customer_id,Purchased
1,Yes
2,Yes
3,Yes
5,Yes
4,Yes
6,Yes
7,Yes
8,Yes
9,Yes
10,No


In [0]:
# Full join
customers_df.join(purchases_df,customers_df['customer_id']==purchases_df['customer_id'],'outer').display()

customer_id,name,age,country,purchase_id,customer_id,item,price
1,Alice,30,USA,101,1,Laptop,1200
1,Alice,30,USA,110,1,Smartwatch,250
2,Bob,25,Canada,102,2,Smartphone,800
3,Cathy,35,UK,103,3,Tablet,600
4,David,28,India,104,4,Headphones,200
5,Eve,40,Germany,105,5,Monitor,300
6,Frank,32,Australia,106,6,Keyboard,100
7,Grace,29,France,107,7,Mouse,50
8,Hank,33,USA,108,8,Printer,150
9,Ivy,27,India,109,9,Camera,500


In [0]:
# Implementing corss join - Most expensive. 10*10 = 100 records
customers_df.crossJoin(purchases_df).display()

customer_id,name,age,country,purchase_id,customer_id,item,price
1,Alice,30,USA,101,1,Laptop,1200
1,Alice,30,USA,102,2,Smartphone,800
1,Alice,30,USA,103,3,Tablet,600
1,Alice,30,USA,104,4,Headphones,200
1,Alice,30,USA,105,5,Monitor,300
1,Alice,30,USA,106,6,Keyboard,100
1,Alice,30,USA,107,7,Mouse,50
1,Alice,30,USA,108,8,Printer,150
1,Alice,30,USA,109,9,Camera,500
1,Alice,30,USA,110,1,Smartwatch,250
